In [ ]:
# download the weaviate client
%pip install -U weaviate-client

In [ ]:
import weaviate, os
from weaviate.config import AdditionalConfig, Timeout
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

# Retrieve environment variables
CLUSTER_URL = os.getenv("CLUSTER_URL")
API_KEY = os.getenv("API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Connect to Weaviate
client = weaviate.connect_to_weaviate_cloud(
	cluster_url=CLUSTER_URL,
	auth_credentials=weaviate.auth.AuthApiKey(API_KEY),
	headers={
		"X-OpenAI-Api-Key": OPENAI_API_KEY,
		"X-Cohere-Api-Key": COHERE_API_KEY,
        "X-Goog-Api-Key": GOOGLE_API_KEY
	},
	additional_config=AdditionalConfig(
		timeout=Timeout(init=30, query=60, insert=120)
	)
)

ready = client.is_ready()
server_version = client.get_meta()["version"]
client_version = weaviate.__version__
live = client.is_live()
connected = client.is_connected()

print(f"Weaviate Ready: {ready}")
print(f"Weaviate Client Version: {client_version}")
print(f"Weaviate Server Version: {server_version}")
print(f"Weaviate Live: {client.is_live()}")
print(f"Client Connected: {connected}")

In [ ]:
# Get schema using REST API
import requests

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

def get_schema():
    resp = requests.get(f"{CLUSTER_URL}/v1/schema", headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

def get_collection_schema(class_name: str):
    resp = requests.get(f"{CLUSTER_URL}/v1/schema/{class_name}", headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

print(get_schema())
print(get_collection_schema("<COLLECTION_NAME>"))

In [ ]:
# Get Meta using REST API 

import requests

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

def get_meta():
    resp = requests.get(f"{CLUSTER_URL}/v1/meta", headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

print(get_meta())

In [ ]:
# Get Nodes using REST API

import requests

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}
def get_nodes(output: str = "minimal"):
    params = {"output": output}
    resp = requests.get(f"{CLUSTER_URL}/v1/nodes", headers=HEADERS, params=params)
    resp.raise_for_status()
    return resp.json()

print(get_nodes())

def get_nodes_for_collection(class_name: str, output: str = "minimal", shard_name: str | None = None):
    params = {"output": output}
    if shard_name:
        params["shardName"] = shard_name
    resp = requests.get(f"{CLUSTER_URL}/v1/nodes/{class_name}", headers=HEADERS, params=params)
    resp.raise_for_status()
    return resp.json()

print(get_nodes_for_collection("Tenants"))

In [ ]:
# Get Shards using REST API

import requests

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

def get_shards(class_name: str, tenant: str | None = None):
    params = {}
    if tenant:
        params["tenant"] = tenant
    resp = requests.get(
        f"{CLUSTER_URL}/v1/schema/{class_name}/shards",
        headers=HEADERS,
        params=params,
    )
    resp.raise_for_status()
    return resp.json()

print(get_shards("Shahin"))

In [ ]:
# Get Cluster Stats using REST API

import requests

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

def get_cluster_stats():
    response = requests.get(
        f"{CLUSTER_URL}/v1/cluster/statistics",
        headers=HEADERS
    )
    response.raise_for_status() 
    return response.json()

print(get_cluster_stats())